In [ ]:
%pip install ultralytics
%pip install supervision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 12.9 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import cv2
import os
from glob import glob
import numpy as np
import pandas as pd

# from deep_sort_realtime.deepsort_tracker import DeepSort

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# Strep no 2 : Parameter
VIDEO_PATH = "/content/drive/MyDrive/ourDataset/video_annotate/cropped_videos/s8.mp4"

BASE_DIR = "/content/Frames/s8_frames/"
VIDEO_OUT_DIR = "/content/s8_video"

# Updated detection model
MODEL_NAME = "yolov9e.pt"

TRACKER = "botsort.yaml"

FIXED_SIZE = 224

CONF_THRESHOLD = 0.35
IOU_THRESHOLD = 0.5

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(VIDEO_OUT_DIR, exist_ok=True)


In [ ]:
# # Step 3: Person tracking fixed frames  (new)

model = YOLO(MODEL_NAME)

cap = cv2.VideoCapture(VIDEO_PATH)

video_fps = int(cap.get(cv2.CAP_PROP_FPS))
print("Original FPS:", video_fps)

tracker_to_student = {}
student_counter = 0

while cap.isOpened():

    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(
        frame,
        persist=True,
        classes=[0],
        tracker=TRACKER,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD
    )

    if results[0].boxes.id is not None:

        for box in results[0].boxes:

            track_id = int(box.id)

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            if track_id not in tracker_to_student:

                student_counter += 1
                tracker_to_student[track_id] = student_counter

                os.makedirs(
                    f"{BASE_DIR}/s8_student_{student_counter}",
                    exist_ok=True
                )

            student_id = tracker_to_student[track_id]

            h, w, _ = frame.shape

            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(w, x2)
            y2 = min(h, y2)

            crop = frame[y1:y2, x1:x2]

            if crop.size == 0:
                continue

            crop = cv2.resize(crop, (FIXED_SIZE, FIXED_SIZE))

            frame_id = int(cap.get(cv2.CAP_PROP_POS_FRAMES))

            cv2.imwrite(
                f"{BASE_DIR}/s8_student_{student_id}/{frame_id}.jpg",
                crop
            )

cap.release()

print("Total Students Detected:", student_counter)

Original FPS: 25

0: 640x608 1 person, 5017.8ms
Speed: 3.6ms preprocess, 5017.8ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 608)

0: 640x608 1 person, 7149.2ms
Speed: 4.5ms preprocess, 7149.2ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 608)

0: 640x608 1 person, 7194.1ms
Speed: 3.7ms preprocess, 7194.1ms inference, 8.0ms postprocess per image at shape (1, 3, 640, 608)

0: 640x608 1 person, 12532.2ms
Speed: 14.7ms preprocess, 12532.2ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 608)

0: 640x608 1 person, 7430.8ms
Speed: 3.6ms preprocess, 7430.8ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 608)

0: 640x608 1 person, 6596.9ms
Speed: 14.1ms preprocess, 6596.9ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 608)

0: 640x608 1 person, 4557.2ms
Speed: 4.5ms preprocess, 4557.2ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 608)

0: 640x608 1 person, 4570.0ms
Speed: 4.0ms preprocess, 4570.0ms infe

In [ ]:
# Step 4: Convert frames to videos (simple & correct)

for student_folder in sorted(os.listdir(BASE_DIR)):

    student_path = os.path.join(BASE_DIR, student_folder)

    if not os.path.isdir(student_path):
        continue

    frames = sorted(
        glob(os.path.join(student_path, "*.jpg")),
        key=lambda x: int(os.path.basename(x).split(".")[0])
    )

    if len(frames) == 0:
        continue

    first = cv2.imread(frames[0])
    h, w, _ = first.shape

    out_path = os.path.join(VIDEO_OUT_DIR, f"{student_folder}.mp4")

    # 🔥 USE ORIGINAL FPS
    out = cv2.VideoWriter(
        out_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        video_fps,
        (w, h)
    )

    for f in frames:
        img = cv2.imread(f)
        if img is not None:
            out.write(img)

    out.release()
    print(f"Saved: {out_path}")

print("All student videos created")

Saved: /content/s10_video/s10_student_1.mp4
Saved: /content/s10_video/s10_student_2.mp4
All student videos created


In [ ]:
# Encode Properly to upload on roboflow or any plateform
!sudo apt-get install ffmpeg

# !mkdir -p '/content/drive/MyDrive/FYP/student_DSC_0251_crop_video'

!for v in "/content/s8_video"/*.mp4; do \
  name=$(basename "$v"); \
  ffmpeg -y -i "$v" \
  -c:v libx264 \
  -pix_fmt yuv420p \
  -movflags +faststart \
  "/content/drive/MyDrive/ourDataset/video_annotate/cropped_videos_new/$name"; \
done

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --

In [ ]:
!cp -r /content/s5_video/ /content/drive/MyDrive/ourDataset/video_annotate/cropped_videos_new/

# Fast process by loop

In [ ]:
%pip install ultralytics
%pip install supervision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 7.3 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import cv2
import os
from glob import glob
import numpy as np

# -----------------------------
# PARAMETERS
# -----------------------------

MODEL_NAME = "yolov9e.pt"
TRACKER = "botsort.yaml"

CONF_THRESHOLD = 0.45
IOU_THRESHOLD = 0.5
FIXED_SIZE = 224

# List of videos to process
VIDEO_LIST = [
# "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0122.MOV",
# "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0123.MOV",
# "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0124.MOV",
# "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0125.MOV",
# "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0126.MOV",
# "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0127.MOV",
# "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0122.MOV",
# "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0128.MOV",
# "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0129.MOV",
# "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0130.MOV",
# "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0131.MOV",
# "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0132.MOV",
# "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0133.MOV",
# "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0134.MOV",
"/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0135.MOV",
"/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0136.MOV",
"/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0137.MOV",
"/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0138.MOV",
"/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0139.MOV",


]

# Output root
OUTPUT_ROOT = "/content/drive/MyDrive/ourDataset/video_annotate/cropped_videos_new"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# -----------------------------
# LOAD MODEL
# -----------------------------

model = YOLO(MODEL_NAME)

# -----------------------------
# PROCESS EACH VIDEO
# -----------------------------

for VIDEO_PATH in VIDEO_LIST:

    video_name = os.path.splitext(os.path.basename(VIDEO_PATH))[0]

    print("\n=================================")
    print("Processing:", video_name)
    print("=================================")

    BASE_DIR = f"/content/Frames/{video_name}_frames"
    VIDEO_OUT_DIR = f"/content/Videos/{video_name}_video"

    os.makedirs(BASE_DIR, exist_ok=True)
    os.makedirs(VIDEO_OUT_DIR, exist_ok=True)

    cap = cv2.VideoCapture(VIDEO_PATH)

    video_fps = int(cap.get(cv2.CAP_PROP_FPS))
    print("FPS:", video_fps)

    tracker_to_student = {}
    student_counter = 0

    # -----------------------------
    # FRAME PROCESSING
    # -----------------------------

    while cap.isOpened():

        ret, frame = cap.read()
        if not ret:
            break

        results = model.track(
            frame,
            persist=True,
            classes=[0],
            tracker=TRACKER,
            conf=CONF_THRESHOLD,
            iou=IOU_THRESHOLD
        )

        if results[0].boxes.id is not None:

            for box in results[0].boxes:

                track_id = int(box.id)

                x1, y1, x2, y2 = map(int, box.xyxy[0])

                if track_id not in tracker_to_student:

                    student_counter += 1
                    tracker_to_student[track_id] = student_counter

                    os.makedirs(
                        f"{BASE_DIR}/{video_name}_student_{student_counter}",
                        exist_ok=True
                    )

                student_id = tracker_to_student[track_id]

                h, w, _ = frame.shape

                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(w, x2)
                y2 = min(h, y2)

                crop = frame[y1:y2, x1:x2]

                if crop.size == 0:
                    continue

                crop = cv2.resize(crop, (FIXED_SIZE, FIXED_SIZE))

                frame_id = int(cap.get(cv2.CAP_PROP_POS_FRAMES))

                cv2.imwrite(
                    f"{BASE_DIR}/{video_name}_student_{student_id}/{frame_id}.jpg",
                    crop
                )

    cap.release()

    print("Students detected:", student_counter)

    # -----------------------------
    # FRAMES → VIDEO
    # -----------------------------

    for student_folder in sorted(os.listdir(BASE_DIR)):

        student_path = os.path.join(BASE_DIR, student_folder)

        if not os.path.isdir(student_path):
            continue

        frames = sorted(
            glob(os.path.join(student_path, "*.jpg")),
            key=lambda x: int(os.path.basename(x).split(".")[0])
        )

        if len(frames) == 0:
            continue

        first = cv2.imread(frames[0])
        h, w, _ = first.shape

        out_path = os.path.join(VIDEO_OUT_DIR, f"{student_folder}.mp4")

        out = cv2.VideoWriter(
            out_path,
            cv2.VideoWriter_fourcc(*"mp4v"),
            video_fps,
            (w, h)
        )

        for f in frames:

            img = cv2.imread(f)

            if img is not None:
                out.write(img)

        out.release()

        print("Saved:", out_path)

    print("Videos created for", video_name)

    # -----------------------------
    # ENCODE VIDEOS
    # -----------------------------

    os.system("sudo apt-get install -y ffmpeg")

    encoded_dir = f"{OUTPUT_ROOT}/{video_name}"
    os.makedirs(encoded_dir, exist_ok=True)

    for v in glob(f"{VIDEO_OUT_DIR}/*.mp4"):

        name = os.path.basename(v)

        os.system(
            f'ffmpeg -y -i "{v}" '
            f'-c:v libx264 '
            f'-pix_fmt yuv420p '
            f'-movflags +faststart '
            f'"{encoded_dir}/{name}"'
        )

    print("Encoded videos saved to:", encoded_dir)

print("\n✅ ALL VIDEOS PROCESSED")



Streaming output truncated to the last 5000 lines.
Speed: 5.0ms preprocess, 43.8ms inference, 2.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 persons, 55.0ms
Speed: 4.8ms preprocess, 55.0ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 persons, 45.3ms
Speed: 3.0ms preprocess, 45.3ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 persons, 61.7ms
Speed: 8.3ms preprocess, 61.7ms inference, 3.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 persons, 79.8ms
Speed: 5.5ms preprocess, 79.8ms inference, 4.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 persons, 48.2ms
Speed: 5.3ms preprocess, 48.2ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 persons, 49.0ms
Speed: 6.1ms preprocess, 49.0ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 persons, 60.0ms
Speed: 6.3ms preprocess, 60.0ms inference, 2.0ms post